# 4.1 · Standardising an ontology language

Before OWL there were many ontology languages (OBO, F-logic, KL-ONE…), causing interoperability problems. The W3C standardised **OWL** (2004), influenced by SHOE, DAML-ONT, OIL, DAML+OIL and 20 years of DL research.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))  # repo paths
sys.path.insert(0, '.')
import matplotlib; matplotlib.use('Agg')
import ch4_toolkit as ch4
import owlready2, rdflib, owlrl, pandas as pd
print('owlready2', owlready2.VERSION, '| rdflib', rdflib.__version__, '| toolkit ready')

owlready2 0.50 | rdflib 7.6.0 | toolkit ready


## 4.1.1 How to design an ontology language (Figure 4.1)
An iterative 7-step process. Here it is as a Python data structure you can inspect.

In [2]:
for s in ch4.LANGUAGE_DESIGN_PROCESS:
    print(f"Step {s['step']}: {s['name']}")
    for t in s['tasks']:
        print('   ', t)

Step 1: Clarification of Scope and Purpose
    1a. Determine scope, benefits
    1b. Long-term perspective
    1c. Economics, feasibility
Step 2: Analysis of General Requirements
    2a. Determine requirements for modelling and reasoning
    2b. Use case scenarios
    2c. Assign priorities
Step 3: Ontological Analysis
    3a. Assess ontological commitments for the language
    3b. Consider performance trade-offs on features and reasoning
Step 4: Language Specification
    4a. Specify syntax and semantics
    4b. Describe glossary and documentation
    4c. Define metamodel
Step 5: Design of Notation for Modeller
    5a. Create graphical notation / sample diagrams or controlled natural language
    5b. Evaluate notation
Step 6: Development of Modelling Tool
    6a. Create computer-processable format
    6b. Create diagrams/CNL and evaluate notation
    6c. Associate with automated reasoner
Step 7: Evaluation and Refinement
    7a. Define test cases, validate and verify
    7b. Check agai

**OWL's stated design goals** (roughly steps 1–3 of the process):

In [3]:
for i, g in enumerate(ch4.OWL_DESIGN_GOALS, 1):
    print(f'{i}. {g}')

1. Shareable
2. Change over time
3. Interoperability
4. Inconsistency detection
5. Balancing expressivity and complexity
6. Ease of use
7. Compatible with existing standards
8. Internationalisation


## 4.1.2 What makes OWL different from a plain DL?

In [4]:
for d in ch4.OWL_VS_DL:
    print('•', d)

• OWL uses IRI references as names (e.g. .../UniOnto.owl#Student); IRIs can be abbreviated.
• Ontologies are documents (RDF/XML) with owl:imports to import one ontology into another.
• OWL files can carry metadata (version info, creators, ...).
• RDF/XSD datatypes are added for ranges of data properties (string, integer, ...).
• Terminology: DL concept -> class, DL role -> object property, plus data property for attributes.


## 4.1.3 The OWL family, first version (OWL 1 species)

Three species: **OWL Lite** (`SHIF(D)`), **OWL DL** (`SHOIN(D)`), **OWL Full** (not a DL). Lite/DL have a model-theoretic semantics; Full has RDF freedom and is undecidable.

In [5]:
for name, info in ch4.OWL1_SPECIES.items():
    print(f"{name}  —  DL: {info['dl']}  (is a DL: {info['is_dl']})")
    print('   ', info['notes'])
    print('    features:', ', '.join(info['features'][:4]), '...')


OWL Lite  —  DL: SHIF(D)  (is a DL: True)
    Classification hierarchy + simple constraints; unqualified (0/1) number restrictions.
    features: Named classes (A), Named properties (P), Individuals C(o), Property values P(o,a) ...
OWL DL  —  DL: SHOIN(D)  (is a DL: True)
    'Maximal' expressiveness while keeping decidability (the 2004 sweet spot).
    features: All OWL Lite features, Arbitrary number restrictions (0 ≤ n), Property value (∃P.{o}), Enumeration {o1,...,on} ...
OWL Full  —  DL: — (not a DL)  (is a DL: False)
    Very high expressiveness, RDF syntactic freedom, meta-classes, self-modifying; loses decidability. NOT a Description Logic.
    features: Meta-classes, Classes as instances, Full RDF freedom ...


### Table 4.1 — OWL class constructs ↔ DL ↔ example
Rendered as a DataFrame, and then **built for real** with `owlready2`.

In [6]:
ch4.table_4_1_constructs()

,OWL construct,DL notation,Example
0,intersectionOf,C₁ ⊓ … ⊓ Cₙ,Human ⊓ Male
1,unionOf,C₁ ⊔ … ⊔ Cₙ,Doctor ⊔ Lawyer
2,complementOf,¬C,¬Male
3,oneOf,"{o₁, …, oₙ}","{giselle, juan}"
4,allValuesFrom,∀P.C,∀hasChild.Doctor
5,someValuesFrom,∃P.C,∃hasChild.Lawyer
6,value,∃P.{o},∃citizenOf.{RSA}
7,minCardinality,≥ nP,≥ 2 hasChild
8,maxCardinality,≤ nP,≤ 6 enrolledIn


### Table 4.2 — OWL axioms ↔ DL ↔ example

In [7]:
ch4.table_4_2_axioms()

,OWL axiom,DL notation,Example
0,SubClassOf,C₁ ⊑ C₂,Human ⊑ Animal ⊓ Biped
1,EquivalentClasses,C₁ ≡ … ≡ Cₙ,Man ≡ Human ⊓ Male
2,SubPropertyOf,P₁ ⊑ P₂,hasDaughter ⊑ hasChild
3,EquivalentProperties,P₁ ≡ … ≡ Pₙ,cost ≡ price
4,SameIndividual,o₁ = … = oₙ,Comrade_Marx = K_Marx
5,DisjointClasses,Cᵢ ⊑ ¬Cⱼ,Male ⊑ ¬Female
6,DifferentIndividuals,oᵢ ≠ oⱼ,Thabo ≠ Andile
7,inverseOf,P₁ ≡ P₂⁻,hasChild ≡ hasParent⁻
8,transitiveProperty,P⁺ ⊑ P,Trans(ancestor)
9,symmetricProperty,P ≡ P⁻,Sym(connectedTo)


### Building those constructs/axioms in Python
`build_construct_demo()` creates `Man ≡ Human ⊓ Male`, `Professional ≡ Doctor ⊔ Lawyer`, `∀hasChild.Doctor`, `≥2 hasChild`, `Male ⊑ ¬Female`, `Human ⊑ Animal ⊓ Biped`, etc. — and we render them back in DL notation.

In [8]:
demo = ch4.build_construct_demo()
for cls in demo.classes():
    print(ch4.dl_render(cls))


Human ⊑ Thing
Human ⊑ (Animal ⊓ Biped)
Male ⊑ Thing
Female ⊑ Thing
Animal ⊑ Thing
Biped ⊑ Thing
Doctor ⊑ Thing
Lawyer ⊑ Thing
Man ⊑ Thing
Man ≡ (Human ⊓ Male)
Professional ⊑ Thing
Professional ≡ (Doctor ⊔ Lawyer)
NonMale ⊑ Thing
NonMale ≡ ¬Male
ParentOfDoctor ⊑ Thing
ParentOfDoctor ≡ ∃hasChild.Doctor
AllKidsDoctors ⊑ Thing
AllKidsDoctors ≡ ∀hasChild.Doctor
HasTwoChildren ⊑ Thing
HasTwoChildren ≡ ≥2 hasChild.Thing


## Example 4.1 — The African Wildlife Ontology (AWO)

The book's tutorial ontology: 10 classes (Lion, Giraffe, Plant…), object properties `eats` and `is-part-of`, and the axiom *“giraffes eat only leaves”* (`Giraffe ⊑ ∀eats.Leaf`). We build it in Python instead of typing XML.

In [9]:
awo = ch4.build_awo(level=0)
print('classes:', [c.name for c in awo.classes()])
print()
print(ch4.dl_render(awo.Giraffe))

classes: ['Animal', 'Plant', 'PlantPart', 'Leaf', 'Twig', 'Root', 'Herbivore', 'Carnivore', 'Giraffe', 'Lion', 'Plant_with_parts']

Giraffe ⊑ Herbivore
Giraffe ⊑ ∀eats.Leaf


**Listing 4.1 twin** — the `Giraffe` class serialised to the *required* RDF/XML exchange syntax, generated from the Python ontology:

In [10]:
print(ch4.awo_giraffe_owl_snippet(awo))

<owl:Class rdf:about="#Giraffe">
  <rdfs:subClassOf rdf:resource="#Herbivore"/>
  <rdfs:subClassOf>
    <owl:Restriction>
      <owl:onProperty rdf:resource="#eats"/>
      <owl:allValuesFrom rdf:resource="#Leaf"/>
    </owl:Restriction>
  </rdfs:subClassOf>
  <rdfs:comment rdf:datatype="http://www.w3.org/2001/XMLSchema#string">Giraffes are herbivores, and they eat only leaves.</rdfs:comment>
</owl:Class>


### Extend it (AWO v1) and run the reasoner
Add proper parthood, `Impala`, `Warthog`, `RockDassie`, and herbivore/carnivore/omnivore definitions. The book classifies `Carnivore ⊑ Animal` with HermiT; here the OWL 2 RL reasoner materialises the subclass/again from the `Animal ⊓ …` definitions.

In [11]:
awo1 = ch4.build_awo(level=1)
print('classes:', len(list(awo1.classes())))
g, b, a, new = ch4.reason_owlrl(awo1)
print(f'closure {b} -> {a} (+{len(new)} inferred triples)')
# show a few inferred rdfs:subClassOf statements
from rdflib import RDFS
subs = [(s, o) for (s, p, o) in new if p == RDFS.subClassOf]
for s, o in list(subs)[:8]:
    print('  inferred subClassOf:', s.split('#')[-1], '⊑', o.split('#')[-1])

classes: 16


closure 99 -> 436 (+337 inferred triples)
  inferred subClassOf: Leaf ⊑ 3
  inferred subClassOf: 13 ⊑ 15
  inferred subClassOf: Root ⊑ Thing
  inferred subClassOf: Leaf ⊑ Leaf
  inferred subClassOf: Nothing ⊑ 15
  inferred subClassOf: 20 ⊑ Thing
  inferred subClassOf: Nothing ⊑ Nothing
  inferred subClassOf: Omnivore ⊑ 19


> **Note on reasoning.** Full DL classification of, e.g., *which individuals are carnivores* needs a DL reasoner (HermiT). OWL 2 RL (used here) is a sound, scalable rule fragment — perfect for subclass/property propagation, which is what we show.